In [3]:
import matplotlib.pyplot as plt
from pathlib import Path
import numpy as np
import pandas as pd
import os
import re
import gc
import itertools
import joblib
import glob

from sklearn.preprocessing import MinMaxScaler
import keras
import keras.backend as K
from keras.models import Model
from keras.layers import Input, LSTM, RepeatVector, TimeDistributed, Dense, Dropout
from keras.optimizers import Adam
from keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split
from pathlib import Path
import seaborn as sns
from sklearn.metrics import auc

2025-06-18 11:31:03.392554: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [4]:
TABLE_PATH = 'position_table.csv'
SCALER_PATH = 'scalers'
THRESHOLDS_PATH = 'thresholds'
TRAIN_FILEPATH = 'data_for_model/sound_levels_data.npy'
MODELS_PATH = 'models'
SILENT_EVENT_HOURS_1 = 'data_for_model/25_event_day.npy'
ONLY_INTRUSIONS = 'data_for_model/complete_intrusions.npy'
LOCATION = 6640

# Helper Functions

In [22]:
def choose_locations(table_path, position):

    position_table = pd.read_csv(table_path)

    selected = position_table[(position_table['location_name']=='quiet') & (~position_table['position_fiber'].isin(position))]
    selected = np.sort(selected.sample(n=100, random_state=12)['position_fiber'].values)

    return selected

In [6]:
def load_normalise_one_loc(path, location, scaler = None):

    data = np.load(path)

    loc = int((location - 1260)/10)
    data = data[:, loc, :]
    shape = data.shape
    flat_values = data.reshape(-1, 1)
    print(f"MIN:{np.min(flat_values)}, MAX:{np.max(flat_values)}")

    if scaler is not None:
        X_normalised = scaler.transform(flat_values)

    else:
        scaler = MinMaxScaler()
        X_normalised = scaler.fit_transform(flat_values)
        
    X_normalised = X_normalised.reshape(shape[0], shape[1], 1)


    return X_normalised, scaler

In [7]:
def load_test_one_loc(test_path, scaler, location, intrusions_only=False):

    loc = int((location - 1260)/10)
    data = np.load(test_path, allow_pickle=True)
    X_data = np.stack(data['data'], axis=0)

    if intrusions_only:
        X_file_refs = np.column_stack((data['row'], data['poi']))
        X_data = X_data[:, loc, :]
    else:
        X_file_refs = data['filename']
        if X_data.ndim == 4:
            X_data = X_data[:, loc, :, :]
        elif X_data.ndim == 3:
            X_data = X_data[:, loc, :]
    
    shape = X_data.shape

    flat_values = X_data.reshape(-1, 1)
    X_normalised = scaler.transform(flat_values)
    X_normalised = X_normalised.reshape(shape[0], shape[1], 1)

    return X_normalised, X_file_refs

In [8]:
def get_performance(metadata, threshold, harsh_testing=True):
    TP, TN, FP, FN = 0, 0, 0, 0


    for _, row in metadata.iterrows():
        detected = row['maxAE'] > threshold
        true_label = row['true_label']
        includes_location = int(row['includes_location'])

        if (true_label == 1) and (includes_location == 1):
            if detected:
                TP += 1
            else:
                FN += 1
        
        elif (true_label) == 1 and (includes_location == 0):
            if detected:
                if harsh_testing:
                    FP += 1
                else:
                    TP +=1
            else:
                TN += 1
        
        elif true_label == 0:
            if detected: 
                FP +=1
            else:
                TN +=1


    return TP, FP, FN, TN

In [9]:
def get_performance_on_negative_class(metadata, threshold):
    TN, FP = 0, 0
    for _, row in metadata.iterrows():
        detected = row['maxAE'] > threshold
        true_label = row['true_label']
        if true_label == 0:
            if detected: 
                FP +=1
            else:
                TN +=1

    return FP, TN

In [10]:
def plot_original_reconstructed_one_loc(location, original_series, reconstructed_series, save_path=None):

    plt.figure(figsize=(12, 5))
    plt.plot(original_series, label="Original", linestyle="--", marker="o", color="royalblue")
    plt.plot(reconstructed_series, label="Reconstructed", linestyle="-", marker="s", color="darkorange")

    plt.xlabel("Time Step")
    plt.ylabel("Feature Value")
    plt.ylim((0, 1))
    plt.title(f"Original vs. Reconstructed Series (Location {location})")
    plt.legend()
    plt.grid(True, linestyle="--", alpha=0.6)
    
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        plt.close()  
    else:
        plt.show()

In [11]:
def predict_on_data(model, data):
    decoded = model.predict(data) 
    absolute_errors = np.abs(decoded - data)
    X_decoded_max = np.max(absolute_errors , axis=1).squeeze()
    return decoded, X_decoded_max

In [12]:
def plot_stacked_bars(false_positives, true_negatives, location_labels, save_path, data_type):
    x = np.arange(len(location_labels))  
    plt.figure(figsize=(12, 6))
    plt.bar(x, false_positives, label="False Positives", color="darkorange")
    plt.bar(x, true_negatives, bottom=false_positives, label="True Negatives", color="royalblue")
    plt.xticks(x, location_labels, rotation=90, fontsize=8, ha='center')
    plt.xlabel("Location ID")
    plt.ylabel("Sample Count")
    plt.title(f"Model Performance on Silent Samples per Location ({data_type} Hours)")
    plt.legend()
    plt.grid(axis='y', linestyle='--', alpha=0.5)
    plt.tight_layout()  
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        plt.close()  
    else:
        plt.show()

In [13]:
def ensure_dir(path):
    """Ensure the directory exists."""
    Path(path).mkdir(parents=True, exist_ok=True)

In [14]:
def save_top_reconstruction_plots(df, X, X_decoded, location, position, label, category, top_n=3):
    """Save top smallest and largest AE plots."""
    top_low_fp = df.sort_values(by='maxAE', ascending=True).head(top_n).index
    top_worst_fp = df.sort_values(by='maxAE', ascending=False).head(top_n).index

    for idx in top_low_fp:
        save_path = f"plots/smallest_false_positives/{position}/Loc_{location}_SoftFP_{category}_{label}_{idx}.png"
        ensure_dir(os.path.dirname(save_path))
        plot_original_reconstructed_one_loc(location, X[idx], X_decoded[idx], save_path)

    for idx in top_worst_fp:
        save_path = f"plots/worst_false_positives/{position}/Loc_{location}_WorstFP_{category}_{label}_{idx}.png"
        ensure_dir(os.path.dirname(save_path))
        plot_original_reconstructed_one_loc(location, X[idx], X_decoded[idx], save_path)

In [15]:
def load_normalise_one_loc_split_train_test(train_path, location):

    data = np.load(train_path)

    loc = int((location - 1260)/10)
    data = data[:, loc, :]

    X_train, X_test = train_test_split(data, test_size=0.2, random_state=12)

    print(f"MIN:{np.min(X_train)}, MAX:{np.max(X_train)}")

    X_train_shape = X_train.shape
    X_test_shape = X_test.shape

    scaler = MinMaxScaler()

    X_train_flat_values = X_train.reshape(-1, 1)
    X_train_normalised = scaler.fit_transform(X_train_flat_values)
    X_train_normalised = X_train_normalised.reshape(X_train_shape[0], X_train_shape[1], 1)

    X_test_flat_values = X_test.reshape(-1, 1)
    X_test_normalised = scaler.transform(X_test_flat_values)
    X_test_normalised = X_test_normalised.reshape(X_test_shape[0], X_test_shape[1], 1)

    return X_train_normalised, X_test_normalised, scaler

In [16]:
def return_max(model, X):
    # predict on X_test, predict on Normal hours (25), predict on Intrusions
    X_decoded = model.predict(X)
    X_error = abs(X - X_decoded)
    X_error_max = np.max(X_error, axis=1).squeeze()
    return X_error_max

In [17]:
def compute_roc_auc(metadata, train_errors, harsh_testing=True):
    tpr_list = []
    fpr_list = []

    thresholds = np.percentile(train_errors, np.linspace(0, 100, 1000))

    for threshold in thresholds:
        TP, FP, FN, TN = get_performance(metadata, threshold, harsh_testing=harsh_testing)
        TPR = TP / (TP + FN) if (TP + FN) != 0 else 0
        FPR = FP / (FP + TN) if (FP + TN) != 0 else 0
        tpr_list.append(TPR)
        fpr_list.append(FPR)

    roc_auc = auc(fpr_list, tpr_list)

    youden_index = np.array(tpr_list) - np.array(fpr_list)
    optimal_threshold_index = np.argmax(youden_index)
    optimal_threshold = thresholds[optimal_threshold_index]
    
    optimal_fpr = np.array(fpr_list)[optimal_threshold_index]
    optimal_tpr = np.array(tpr_list)[optimal_threshold_index]

    return {
        'roc_auc': roc_auc,
        'thresholds': thresholds,
        'fprs': fpr_list,
        'tprs': tpr_list,
        'harsh_testing': harsh_testing,
        'optimal_threshold': optimal_threshold,
        'optimal_fpr': optimal_fpr,
        'optimal_tpr': optimal_tpr,
    }

In [18]:
def plot_confusion_matrix(tp, fp, fn, tn, save_path = None, title='Confusion Matrix'):
    cm = np.array([[tp, fn],
                   [fp, tn]])

    labels = ['Positive', 'Negative']

    plt.figure(figsize=(6, 5))
    ax = sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
                     xticklabels=labels, yticklabels=labels,
                     linewidths=0.5, linecolor='gray', square=True)


    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.title(title, fontsize=14, weight='bold')
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        plt.close()  
    else:
        plt.show()

In [19]:
def create_location_plots(df, save_plots=True, show_plots=True):
    """
    Create scatter plots for each location showing Silent FP rate vs Silent TN rate.
    Each point represents a model and is labeled with the model number.
    
    Parameters:
    df (pandas.DataFrame): DataFrame with the model performance data
    save_plots (bool): Whether to save plots as image files
    show_plots (bool): Whether to display plots
    """
    
    # Get unique locations
    locations = sorted(df['Location'].unique())
    
    print(f"Creating plots for {len(locations)} locations...")
    
    # Set up the plot style
    plt.style.use('default')
    
    for location in locations:
        # Filter data for current location
        location_data = df[df['Location'] == location].copy()
        
        # Create figure and axis
        fig, ax = plt.subplots(figsize=(10, 8))
        
        # Extract data for plotting
        x_data = location_data['Silent_FP_rate']
        y_data = location_data['Silent_TN_rate']
        models = location_data['Model']
        
        # Create scatter plot
        scatter = ax.scatter(x_data, y_data, s=100, alpha=0.7, 
                           c='steelblue', edgecolors='black', linewidth=1)
        
        # Add model labels to each point
        for i, (x, y, model) in enumerate(zip(x_data, y_data, models)):
            ax.annotate(str(model), (x, y), 
                       xytext=(3, 3), textcoords='offset points',
                       fontsize=8, fontweight='normal',
                       bbox=dict(boxstyle='round,pad=0.2', 
                               facecolor='white', alpha=0.7))
        
        # Customize the plot
        ax.set_xlabel('Silent False Positive Rate', fontsize=12, fontweight='bold')
        ax.set_ylabel('Silent True Negative Rate', fontsize=12, fontweight='bold')
        ax.set_title(f'Models Performance at Location {location}', 
                    fontsize=14, fontweight='bold', pad=20)
        
        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1)
        # Add grid for better readability
        ax.grid(True, alpha=0.3)
        
        best_score = y_data - x_data  # TN rate - FP rate (maximize this)
        worst_score = x_data - y_data  # FP rate - TN rate (minimize this, so maximize negative)
        
        best_idx = best_score.idxmax()
        worst_idx = worst_score.idxmax()
        
        best_model = location_data.loc[best_idx, 'Model']
        worst_model = location_data.loc[worst_idx, 'Model']
        
        # Add context text with best/worst models
        context_text = f'Models tested: {len(location_data)}\nBest: {best_model}\nWorst: {worst_model}'
        ax.text(0.98, 0.98, context_text, fontsize=9,
                verticalalignment='top', horizontalalignment='right',
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
        
        # Improve layout
        plt.tight_layout()
        
        # Save plot if requested
        ensure_dir("plots/metrics")
        if save_plots:
            filename = f'plots/metrics/location_{location}_performance.png'
            plt.savefig(filename, dpi=300, bbox_inches='tight')
            print(f"Saved plot for location {location}: {filename}")
        
        # Show plot if requested
        if show_plots:
            plt.show()
        else:
            plt.close()

# Validating on 4220

In [ ]:
def validate_on_one_loc():
    for model_name in os.listdir(MODELS_PATH):

        # Load the model that has nasme structure "location_{number}.keras"
        match = re.match(r"location_(\d+)\.keras", model_name)
        if not match:
            print(f"Skipping invalid file: {model_name}")
            continue
        
        model_path = os.path.join(MODELS_PATH, model_name)
        model = keras.models.load_model(model_path)

        # Extract the location from the name
        location = int(match.group(1))

        # Load the training set based on the location 
        X_train, _, scaler = load_normalise_one_loc_split_train_test(TRAIN_FILEPATH, location)

        #  Predict on training data and extract training errors
        decoded = model.predict(X_train)
        train_error = np.abs(decoded - X_train)

        # load the 4220 data (19, 25, and intrusions), normalising with the appropriate scaler
        _, X_test_4220, _ = load_normalise_one_loc_split_train_test(TRAIN_FILEPATH, 4220)
        X_25_4220, _ = load_test_one_loc(SILENT_EVENT_HOURS_1, scaler, 4220)
        X_25_4220 = X_25_4220[:420, :, :]
        X_intrusions_4220, X_intrusions_files_4220 = load_test_one_loc(ONLY_INTRUSIONS, scaler, 4220, True)
        
        # Make predictions
        X_test_4220_max = return_max(model, X_test_4220)
        X_25_4220_max = return_max(model, X_25_4220)
        X_intrusions_4220_max = return_max(model, X_intrusions_4220)

        # create data  frame
        meta_data_19 = pd.DataFrame({'event': np.array([f"Silent_sample_19_{i}" for i in range(X_test_4220_max.shape[0])]), 
                                    'maxAE': X_test_4220_max, 
                                    'true_label': np.zeros(X_test_4220_max.shape[0], dtype=int),
                                    'includes_location': np.ones(X_test_4220_max.shape[0], dtype=int)})
        
        meta_data_25 = pd.DataFrame({'event': np.array([f"Silent_sample_25_{i}" for i in range(X_25_4220_max.shape[0])]), 
                                    'maxAE': X_25_4220_max, 
                                    'true_label': np.zeros(X_25_4220_max.shape[0], dtype=int),
                                    'includes_location': np.ones(X_25_4220_max.shape[0], dtype=int)})
        
        event_row_new = np.copy(X_intrusions_files_4220)
        event_row_new[:, 1] = (np.char.find(X_intrusions_files_4220[:, 1], str(4220)) != -1).astype(int)
        intrusions_meta_data = pd.DataFrame({'event': event_row_new[:, 0], 
                                            'maxAE': X_intrusions_4220_max, 
                                            'true_label': np.ones(X_intrusions_4220_max.shape[0], dtype=int), 
                                            'includes_location': event_row_new[:, 1].astype(int)})
        intrusions_meta_data = intrusions_meta_data.groupby(['event'], as_index=False).max()

        combined_data_frame = pd.concat([meta_data_19, meta_data_25, intrusions_meta_data], ignore_index=True)

        # AUC-ROC method, extract threhsolds
        roc_result_harsh = compute_roc_auc(combined_data_frame, train_error, True)
        roc_result_soft = compute_roc_auc(combined_data_frame, train_error, False)

        threshold_soft = roc_result_soft['optimal_threshold']
        threshold_harsh = roc_result_harsh['optimal_threshold']

        np.savez(f"{THRESHOLDS_PATH}/thresholds_{location}.npz", soft=threshold_soft, harsh=threshold_harsh)

        # print/save the confusion matrix
        ensure_dir("plots/cms")
        TP, FP, FN, TN = get_performance(combined_data_frame, threshold_soft, False)
        plot_confusion_matrix(TP, FP, FN, TN, f"plots/cms/{location}_on_4220.png")

# Testing

In [24]:
def test(position, locations):

    model = keras.models.load_model(f"{MODELS_PATH}/multi_loc_1.keras")
    scaler = joblib.load(F'{SCALER_PATH}/scaler_multi.pkl')
    thresholds = np.load(f'{THRESHOLDS_PATH}/thresholds_multi.npz')

    location_labels = []
    silent_false_positives, silent_true_negatives = [], []
    intrusion_false_positives, intrusion_true_negatives = [], []

    # for each location 
    for location in locations: 
        print(f"Working with location {location}.")
        # if location >= 4200 and <=4300, skip
        if (location >= 4200) & (location <= 4300):
            continue
        # load the 19/02 data
        X_19, _ = load_normalise_one_loc(TRAIN_FILEPATH, location, scaler)

        # load the 29/02 morning data
        test_data_25, _ = load_test_one_loc(SILENT_EVENT_HOURS_1, scaler, location)
        X_25 = test_data_25[:420, :, :]

        # load the intrusions data
        X_intrusions, _ = load_test_one_loc(ONLY_INTRUSIONS, scaler, location, True)

        # predict and extract maxAE values
        X_19_decoded, max_AE_19 = predict_on_data(model, X_19)
        X_25_decoded, max_AE_25 = predict_on_data(model, X_25)
        X_intrusions_decoded, max_AE_intrusions = predict_on_data(model, X_intrusions)

        # create data  frame
        meta_data_19 = pd.DataFrame({'event': np.array([f"Silent_sample_19_{i}" for i in range(max_AE_19.shape[0])]), 
                                    'maxAE': max_AE_19, 
                                    'true_label': np.zeros(max_AE_19.shape[0], dtype=int)})
        
        meta_data_25 = pd.DataFrame({'event': np.array([f"Silent_sample_25_{i}" for i in range(max_AE_25.shape[0])]), 
                                    'maxAE': max_AE_25, 
                                    'true_label': np.zeros(max_AE_25.shape[0], dtype=int)})
        
        all_silent_meta = pd.concat([meta_data_19, meta_data_25], ignore_index=True)
        
        meta_data_intrusions = pd.DataFrame({'event': np.array([f"Intrusion_sample_{i}" for i in range(max_AE_intrusions.shape[0])]), 
                                    'maxAE': max_AE_intrusions, 
                                    'true_label': np.zeros(max_AE_intrusions.shape[0], dtype=int)})
        

        # get the FP and TN
        threshold = thresholds['soft']
        FP_s, TN_s = get_performance_on_negative_class(all_silent_meta, threshold)
        FP_i, TN_i = get_performance_on_negative_class(meta_data_intrusions, threshold)

        location_labels.append(str(location))  
        silent_false_positives.append(FP_s)
        silent_true_negatives.append(TN_s)

        intrusion_false_positives.append(FP_i) 
        intrusion_true_negatives.append(TN_i)

        silent_19_above_thresh = meta_data_19[meta_data_19['maxAE'] > threshold]
        silent_25_above_thresh = meta_data_25[meta_data_25['maxAE'] > threshold]
        intrusion_above_thresh = meta_data_intrusions[meta_data_intrusions['maxAE'] > threshold]

        save_top_reconstruction_plots(silent_19_above_thresh, X_19, X_19_decoded, location, position, "silent_19", "Silent")
        save_top_reconstruction_plots(silent_25_above_thresh, X_25, X_25_decoded, location, position, "silent_25", "Silent")
        save_top_reconstruction_plots(intrusion_above_thresh, X_intrusions, X_intrusions_decoded, location, position, "intrusion", "Intrusion")

        # Clean up memory
        del X_19, X_25, X_intrusions
        del X_19_decoded, X_25_decoded, X_intrusions_decoded
        del max_AE_19, max_AE_25, max_AE_intrusions
        del meta_data_19, meta_data_25, meta_data_intrusions
        del silent_19_above_thresh, silent_25_above_thresh, intrusion_above_thresh
        gc.collect()



    ensure_dir("plots/performance")
    plot_stacked_bars(silent_false_positives, silent_true_negatives, location_labels, f"plots/performance/Pos_Silent_{position}.png", "Silent")
    plot_stacked_bars(intrusion_false_positives, intrusion_true_negatives, location_labels, f"plots/performance/Pos_Intrusion_{position}.png", "Intrusion")

        # Create a DataFrame with performance metrics
    performance_df = pd.DataFrame({
        'Model': [position] * len(location_labels),
        'Location': location_labels,
        'Silent_TN': silent_true_negatives,
        'Silent_FP': silent_false_positives,
        'Intrusion_TN': intrusion_true_negatives,
        'Intrusion_FP': intrusion_false_positives
    })

    # Calculate rates
    performance_df['Silent_TN_rate'] = performance_df['Silent_TN'] / (performance_df['Silent_TN'] + performance_df['Silent_FP'])
    performance_df['Silent_FP_rate'] = performance_df['Silent_FP'] / (performance_df['Silent_TN'] + performance_df['Silent_FP'])

    performance_df['Intrusion_TN_rate'] = performance_df['Intrusion_TN'] / (performance_df['Intrusion_TN'] + performance_df['Intrusion_FP'])
    performance_df['Intrusion_FP_rate'] = performance_df['Intrusion_FP'] / (performance_df['Intrusion_TN'] + performance_df['Intrusion_FP'])

    # Save the DataFrame to CSV
    os.makedirs("metrics", exist_ok=True)
    csv_path = f"metrics/performance_position_{position}.csv"
    performance_df.to_csv(csv_path, index=False)
    print(f"Performance CSV saved to: {csv_path}")


In [25]:
model = keras.models.load_model(f"{MODELS_PATH}/multi_loc_1.keras")
position = [5860, 3540, 1430, 4350, 4290, 1680, 3800, 2680, 2100]
locations = choose_locations(TABLE_PATH, position)
test(position, locations)

Working with location 1540.
MIN:9.42511957655043, MAX:51.26705119280432
44/44 ━━━━━━━━━━━━━━━━━━━━ 12s 259ms/step
14/14 ━━━━━━━━━━━━━━━━━━━━ 3s 208ms/step
78/78 ━━━━━━━━━━━━━━━━━━━━ 18s 229ms/step
Working with location 1550.
MIN:9.006667059645839, MAX:55.37424517769779
44/44 ━━━━━━━━━━━━━━━━━━━━ 10s 235ms/step
14/14 ━━━━━━━━━━━━━━━━━━━━ 3s 248ms/step
78/78 ━━━━━━━━━━━━━━━━━━━━ 18s 234ms/step
Working with location 1600.
MIN:8.70559915618477, MAX:59.12425358611728
44/44 ━━━━━━━━━━━━━━━━━━━━ 10s 234ms/step
14/14 ━━━━━━━━━━━━━━━━━━━━ 3s 218ms/step
78/78 ━━━━━━━━━━━━━━━━━━━━ 17s 222ms/step
Working with location 1650.
MIN:8.403619048194136, MAX:60.7982946343158
44/44 ━━━━━━━━━━━━━━━━━━━━ 10s 227ms/step
14/14 ━━━━━━━━━━━━━━━━━━━━ 3s 214ms/step
78/78 ━━━━━━━━━━━━━━━━━━━━ 18s 232ms/step
Working with location 1660.
MIN:8.364328883702166, MAX:60.038540525363686
44/44 ━━━━━━━━━━━━━━━━━━━━ 10s 220ms/step
14/14 ━━━━━━━━━━━━━━━━━━━━ 3s 206ms/step
78/78 ━━━━━━━━━━━━━━━━━━━━ 18s 228ms/step
Working with

In [ ]:
for model_name in os.listdir(MODELS_PATH):
    # Load the model that has anme structure "location_{number}.keras"
    match = re.match(r"location_(\d+)\.keras", model_name)
    if not match:
        continue                          
    
    model_path = os.path.join(MODELS_PATH, model_name)
    model = keras.models.load_model(model_path)

    # Extract the location from the name
    position = int(match.group(1))
    already_tested = any(str(position) in fname for fname in os.listdir("metrics"))
    if already_tested:
        print(f"Skipping position {position} - results already exist.")
        continue
    print(f"Model: {position}")
    locations = choose_locations(TABLE_PATH, position)
    test(position, locations)
    del model
    K.clear_session()


In [ ]:
def merge_csv_files(directory_path, output_filename='merged_data.csv'):
    """
    Merge all CSV files in a directory into one DataFrame and save as CSV.
    
    Parameters:
    directory_path (str): Path to directory containing CSV files
    output_filename (str): Name for the output merged CSV file
    """
    
    # Get all CSV files in the directory
    csv_pattern = os.path.join(directory_path, "*.csv")
    csv_files = glob.glob(csv_pattern)
    
    if not csv_files:
        print(f"No CSV files found in directory: {directory_path}")
        return
    
    print(f"Found {len(csv_files)} CSV files:")
    for file in csv_files:
        print(f"  - {os.path.basename(file)}")
    
    # Read and combine all CSV files
    dataframes = []
    
    for file in csv_files:
        try:
            df = pd.read_csv(file)
            dataframes.append(df)
        except Exception as e:
            print(f"Error reading {file}: {e}")
    
    if not dataframes:
        print("No valid CSV files could be loaded.")
        return
    
    # Combine all dataframes
    merged_df = pd.concat(dataframes, ignore_index=True)
    
    # Save the merged dataframe
    output_path = os.path.join(directory_path, output_filename)
    merged_df.to_csv(output_path, index=False)
    
    print(f"\nMerged data saved to: {output_path}")
    print(f"Total rows: {len(merged_df)}")
    print(f"Total columns: {len(merged_df.columns)}")
    print(f"Columns: {list(merged_df.columns)}")
    

In [ ]:
merge_csv_files("metrics", output_filename='merged_data.csv')

In [17]:
df = pd.read_csv('metrics/merged_data.csv')
create_location_plots(df, True, False)

Creating plots for 174 locations...
Saved plot for location 1540: plots/metrics/location_1540_performance.png
Saved plot for location 1550: plots/metrics/location_1550_performance.png
Saved plot for location 1560: plots/metrics/location_1560_performance.png
Saved plot for location 1600: plots/metrics/location_1600_performance.png
Saved plot for location 1610: plots/metrics/location_1610_performance.png
Saved plot for location 1650: plots/metrics/location_1650_performance.png
Saved plot for location 1660: plots/metrics/location_1660_performance.png
Saved plot for location 1670: plots/metrics/location_1670_performance.png
Saved plot for location 1680: plots/metrics/location_1680_performance.png
Saved plot for location 1690: plots/metrics/location_1690_performance.png
Saved plot for location 1700: plots/metrics/location_1700_performance.png
Saved plot for location 1710: plots/metrics/location_1710_performance.png
Saved plot for location 1750: plots/metrics/location_1750_performance.png
Sa